# 1. Initial RAG PoC for AION
**Goal:** Create a basic Retrieval-Augmented Generation (RAG) pipeline to answer questions based on OCPP specifications and sample logs.

### Step 1: Install and Load Dependencies

### Step 2: Load Data
Load documents from the `data/` directory. For this PoC, we'll use both sample logs and OCPP specs.

In [7]:
# Load documents from multiple directories using TextLoader for simple files
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

data_dirs = ["../data/sample_logs", "../data/ocpp_spec"]
documents = []

for data_path in data_dirs:
    data_path_obj = Path(data_path)
    if data_path_obj.exists():
        # Get all files in the directory
        for file_path in data_path_obj.rglob("*"):
            if file_path.is_file() and file_path.suffix in ['.log', '.txt', '.md'] and file_path.name != '.gitkeep':
                try:
                    loader = TextLoader(str(file_path), encoding='utf-8')
                    docs = loader.load()
                    documents.extend(docs)
                    print(f"Loaded {len(docs)} documents from {file_path}")
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")

print(f"Total documents loaded: {len(documents)}")
if documents:
    print(f"Sample document content: {documents[0].page_content[:200]}...")
else:
    print("No documents found")

Loaded 1 documents from ../data/sample_logs/successful_session.log
Loaded 1 documents from ../data/sample_logs/charging_session_error.log
Loaded 1 documents from ../data/ocpp_spec/ocpp_2.0.1_charging_management.md
Total documents loaded: 3
Sample document content: 2024-11-02 11:30:15 [INFO] Charging session started for connector 2
2024-11-02 11:30:15 [INFO] Station ID: CP001, Connector: 2, RFID: 9876543210
2024-11-02 11:30:16 [INFO] Vehicle connected, authentic...


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
print(f"Created {len(splits)} document chunks")
print(f"Sample chunk: {splits[0].page_content[:200] if splits else 'No chunks created'}")

Created 4 document chunks
Sample chunk: 2024-11-02 11:30:15 [INFO] Charging session started for connector 2
2024-11-02 11:30:15 [INFO] Station ID: CP001, Connector: 2, RFID: 9876543210
2024-11-02 11:30:16 [INFO] Vehicle connected, authentic


In [9]:
# Use a completely offline approach with TF-IDF embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Using offline TF-IDF embeddings (no internet required)")

class TFIDFEmbeddings:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words='english',
            ngram_range=(1, 2)
        )
        self.is_fitted = False
    
    def embed_documents(self, texts):
        if not self.is_fitted:
            # Fit the vectorizer on all documents
            self.vectorizer.fit(texts)
            self.is_fitted = True
        
        # Transform texts to TF-IDF vectors
        vectors = self.vectorizer.transform(texts)
        return vectors.toarray().tolist()
    
    def embed_query(self, text):
        if not self.is_fitted:
            raise ValueError("Must embed documents first")
        
        vector = self.vectorizer.transform([text])
        return vector.toarray()[0].tolist()

# Create our custom embedding function
embedding_function = TFIDFEmbeddings()

# Use FAISS for vector storage (works offline)
from langchain_community.vectorstores import FAISS

try:
    vectorstore = FAISS.from_documents(splits, embedding_function)
    print("✅ FAISS vector store created successfully with TF-IDF embeddings!")
    print("📡 This works completely offline - no internet connection needed")
except Exception as e:
    print(f"❌ Error creating vector store: {e}")
    # Ultimate fallback - simple text search
    print("🔍 Using simple text matching as final fallback...")
    
    class SimpleTextSearch:
        def __init__(self, documents):
            self.documents = documents
        
        def as_retriever(self, **kwargs):
            return SimpleRetriever(documents=self.documents, k=kwargs.get('search_kwargs', {}).get('k', 5))
    
    class SimpleRetriever(BaseRetriever):
        def __init__(self, documents, k=5):
            self.documents = documents
            self.k = k
        
        def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
            # Simple keyword matching
            query_words = query.lower().split()
            scored_docs = []
            
            for doc in self.documents:
                content = doc.page_content.lower()
                score = sum(1 for word in query_words if word in content)
                if score > 0:
                    scored_docs.append((score, doc))
            
            # Sort by score and return top k
            scored_docs.sort(key=lambda x: x[0], reverse=True)
            return [doc for _, doc in scored_docs[:self.k]]
    
    vectorstore = SimpleTextSearch(splits)

Using offline TF-IDF embeddings (no internet required)
❌ Error creating vector store: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).
🔍 Using simple text matching as final fallback...


In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Using your available Llama3:8b model
llm = Ollama(model="llama3:8b")
print("Retriever and LLM setup complete!")

Retriever and LLM setup complete!


/var/folders/gx/k1m3rp6d757_skvbfw3l_ndc0000gn/T/ipykernel_77961/103001267.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3:8b")


In [11]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain created successfully!")

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class '__main__.SimpleRetriever'>

In [18]:
# Try additional questions
questions = [
    "What are the different connector states in OCPP?",
    "How does thermal protection work in charging stations?",
    "What error code indicates ground fault protection?",
    "What happened to connector 1 in the logs?"
]

for q in questions:
    print(f"\n--- Question: {q} ---")
    try:
        # Use .stream() for better real-time output in notebooks
        full_response = ""
        for chunk in rag_chain.stream(q):
            print(chunk, end="", flush=True)
            full_response += chunk
        # print(f"Answer: {full_response}") # Optional: print full response at the end
    except Exception as e:
        print(f"Error: {str(e)}")


--- Question: What are the different connector states in OCPP? ---
Answer: Based on the provided context, I can answer that according to the OCPP 2.0.1 specification, there are five different connector states:

1. **AVAILABLE**: Ready for charging
2. **OCCUPIED**: Vehicle connected but not charging
3. **CHARGING**: Active charging session
4. **FAULTED**: Error condition requires maintenance
5. **UNAVAILABLE**: Temporarily out of service

--- Question: How does thermal protection work in charging stations? ---
Answer: Based on the provided context, I can answer that according to the OCPP 2.0.1 specification, there are five different connector states:

1. **AVAILABLE**: Ready for charging
2. **OCCUPIED**: Vehicle connected but not charging
3. **CHARGING**: Active charging session
4. **FAULTED**: Error condition requires maintenance
5. **UNAVAILABLE**: Temporarily out of service

--- Question: How does thermal protection work in charging stations? ---
Answer: According to the provided co

### 🔧 Fix for Garbled Responses

The issue with garbled/mixed responses occurs because our simple retriever has problems with:
1. **Context mixing**: Different questions get similar contexts.
2. **LLM state**: Previous responses might influence new ones.
3. **Simple matching**: Keyword matching isn't sophisticated enough.